# 69 — Held-out PRO160 Q10/Q50 analysis and uncertainty-gate screening

This notebook analyzes the exact three-arm cohort produced by workers 68:

- stock VLA: 10 Euler steps, execute 10 actions;
- Q10 planner: 64 candidates, top-16 Q-softmax blend, 3 Euler steps, execute 10;
- Q50 planner: the same planner, trained/scored on the 50-action horizon.

Every reported SR delta uses **all exact matched identities in its stated denominator**.
The first section uses all 160 held-out identities. The true P&P U10/U20/U50 section
uses the 140 identities that overlap the earlier worker-41 no-op diagnostic; the x0.3
suite was not in that older cohort.

Important: workers 68 did **not** persist P&P U10/U20/U50 or per-action-dimension U.
They did persist Q distributions and first-10/full-50 candidate diversity. This notebook
keeps those signals separate and never calls Q spread "P&P uncertainty." First-chunk
gate sweeps are episode-start screening proxies; multi-chunk/whole-episode sweeps are
post-hoc diagnostics and are not claims about an online switching policy.


In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())


## Configuration and imports

`REQUIRE_FULL_COHORT=True` means the full **planned held-out cohort of 160 identities**,
not the older 1,300-episode cohort. Set it to `False` only for a preliminary look while
a worker is still finishing; only identities complete under all three arms are retained.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm

from analysis.horizon_diagnostics import (
    HORIZONS, load_horizon_artifacts, prefix_failure_auc_table,
    prefix_feature_table, quantile_outcome_curve,
    validate_diagnostic_cohort)
from analysis.qplanning_heldout import (
    ARM_LABELS, Q_SIGNALS, attach_signal_to_pair, discordant_selector_auc,
    extract_qplanning_boundaries, failure_auc_table, paired_effect_tables,
    q_chunk_auc_table, q_episode_features, q_prefix_auc_table,
    q_prefix_features, quantile_gate_sweep, success_tables,
    validate_qplanning_heldout_cohort)
from pnp.config import Method, RolloutConfig
from pnp.diversity import (
    DIVERSITY_PAIR_KEYS, SOURCE_HORIZON_MULTI_QUERY_EXPERIMENT)
from pnp.qplanning_eval_experiment import (
    QPLANNING_HELDOUT_EXPERIMENT, QPLANNING_HELDOUT_IDENTITIES)
from pnp.store import SupabaseStore

REQUIRE_FULL_COHORT = True
INCLUDE_WORKER41_U = True
EXPECTED_U_OVERLAP = 140
N_BOOT = 2000
GATE_GRID_SIZE = 21
MIN_GATE_EPISODES = 10
OUTPUT = Path('qplanning_heldout160_analysis_outputs')
CACHE = OUTPUT / 'cache'
OUTPUT.mkdir(exist_ok=True); CACHE.mkdir(exist_ok=True)
store = SupabaseStore()


## Load and audit the exact matched cohort

The audit rejects duplicate identities and mixed behavior hashes. All later SR comparisons
use the intersection complete under stock, Q10, and Q50.


In [ ]:
rows = pd.DataFrame(store.fetch_all(
    'rollouts', '*', configure=lambda query: query.eq(
        'experiment', QPLANNING_HELDOUT_EXPERIMENT),
    order_by=('rollout_id',)))
arms = validate_qplanning_heldout_cohort(
    rows, expected_identities=QPLANNING_HELDOUT_IDENTITIES,
    require_complete=REQUIRE_FULL_COHORT)
n_matched = len(arms[Method.VANILLA])
print(f'Exact complete stock/Q10/Q50 identities: {n_matched}/160')

audit = []
for method, arm in arms.items():
    config = arm.config_json.iloc[0]
    if isinstance(config, str):
        import json
        config = json.loads(config)
    audit.append({
        'arm': ARM_LABELS[method], 'rows': len(arm),
        'behavior_hash': arm.config_hash.iloc[0],
        'decode_steps': config.get('num_inference_steps'),
        'executed_actions': config.get('n_action_steps'),
        'candidates': config.get('num_samples'),
        'Q_checkpoint': config.get('qplanning_ckpt_id'),
    })
display(pd.DataFrame(audit))
coverage = (arms[Method.VANILLA].groupby('suite', sort=True)
            .size().rename('matched_identities').reset_index())
display(coverage)


## Primary result: overall and per-suite paired SR

`planner minus stock` is the ordinary difference in success rate over every matched
identity. `F→S` and `S→F` are the paired outcome transitions.


In [ ]:
overall_sr, suite_sr = success_tables(arms)
effects, suite_effects, pairs = paired_effect_tables(arms)
display(overall_sr)
display(effects[[
    'planner', 'episodes', 'baseline_sr_pct', 'condition_sr_pct',
    'condition_minus_baseline_pp', 'delta_ci_low_pp', 'delta_ci_high_pp',
    'failure_to_success', 'success_to_failure', 'paired_p_value']])
overall_sr.to_csv(OUTPUT / 'overall_success.csv', index=False)
suite_sr.to_csv(OUTPUT / 'success_by_suite.csv', index=False)
effects.to_csv(OUTPUT / 'paired_effects_overall.csv', index=False)
suite_effects.to_csv(OUTPUT / 'paired_effects_by_suite.csv', index=False)

suites = sorted(suite_sr.suite.unique())
labels = [suite.removeprefix('libero_') for suite in suites]
x = np.arange(len(suites)); width = .25
colors = {'stock VLA': '#4C78A8', 'Q10 planner': '#F58518',
          'Q50 planner': '#54A24B'}
fig, axes = plt.subplots(2, 1, figsize=(14, 10), constrained_layout=True)
for offset, label in zip((-width, 0, width), colors):
    group = (suite_sr[suite_sr.arm.eq(label)].set_index('suite')
             .reindex(suites))
    axes[0].bar(x + offset, group.success_rate_pct, width,
                label=label, color=colors[label])
axes[0].set_xticks(x, labels, rotation=35, ha='right')
axes[0].set(ylabel='Success rate (%)', ylim=(0, 105),
            title='Held-out LIBERO-PRO SR on exact matched identities')
axes[0].legend(); axes[0].grid(axis='y', alpha=.2)
for offset, planner, color in ((-.15, 'Q10 planner', '#F58518'),
                               (.15, 'Q50 planner', '#54A24B')):
    group = (suite_effects[suite_effects.planner.eq(planner)]
             .set_index('suite').reindex(suites))
    y = group.condition_minus_baseline_pp.to_numpy(float)
    low = y - group.delta_ci_low_pp.to_numpy(float)
    high = group.delta_ci_high_pp.to_numpy(float) - y
    axes[1].bar(x + offset, y, .28, label=f'{planner} minus stock', color=color)
    axes[1].errorbar(x + offset, y, yerr=np.vstack((low, high)),
                     fmt='none', color='black', capsize=2, linewidth=.8)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(x, labels, rotation=35, ha='right')
axes[1].set(ylabel='Paired SR change (percentage points)',
            title='Whole-matched-cohort SR change by suite')
axes[1].legend(); axes[1].grid(axis='y', alpha=.2)
fig.savefig(OUTPUT / 'sr_and_delta_by_suite.png', dpi=180)
plt.show()


## Decode the Q score and candidate-diversity telemetry

The risk-oriented Q signals are `-Q` (larger means the critic is more pessimistic),
Q spread/standard deviation, elite-weight entropy, and action disagreement among the
64 generated candidates. These are **not** the K=5 P&P U values.


In [ ]:
boundaries = extract_qplanning_boundaries(arms)
q_features = q_episode_features(boundaries)
q_prefix = q_prefix_features(boundaries, max_chunks=8)
print({'boundary_rows': len(boundaries), 'episode_rows': len(q_features),
       'mean_boundaries_per_episode': boundaries.groupby('rollout_id').size().mean()})
display(boundaries.groupby('planner')[[
    'q_mean', 'q_std', 'q_range', 'elite_effective_n',
    'first10_pairwise_rms', 'full50_pairwise_rms',
    'inference_ms']].mean())
boundaries.to_csv(OUTPUT / 'q_boundary_telemetry.csv', index=False)
q_features.to_csv(OUTPUT / 'q_episode_features.csv', index=False)


## Do Q scores or candidate diversity predict failure?

AUC > 0.5 means the named risk signal is larger on failures. The first-chunk version
is available before any action is executed; the episode mean is post-hoc.


In [ ]:
q_auc_parts = []
for method in (Method.QPLANNING_Q10, Method.QPLANNING_Q50):
    group = q_features[q_features.method.eq(method)]
    scores = ([f'{signal}_first_chunk' for signal in Q_SIGNALS]
              + [f'{signal}_episode' for signal in Q_SIGNALS])
    table = failure_auc_table(group, scores, n_boot=N_BOOT)
    table.insert(0, 'planner', ARM_LABELS[method])
    q_auc_parts.append(table)
q_auc = pd.concat(q_auc_parts, ignore_index=True)
pooled_q_auc = q_auc[q_auc.suite.eq('pooled')].copy()
display(pooled_q_auc[[
    'planner', 'score_name', 'episodes', 'failures', 'failure_auc',
    'auc_ci_low', 'auc_ci_high']])
q_auc.to_csv(OUTPUT / 'q_signal_failure_auc.csv', index=False)

key_signals = ('negative_q_weighted', 'q_std',
               'first10_pairwise_rms', 'full50_pairwise_rms')
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
for row_idx, planner in enumerate(('Q10 planner', 'Q50 planner')):
    for col_idx, suffix in enumerate(('first_chunk', 'episode')):
        ax = axes[row_idx, col_idx]
        names = [f'{signal}_{suffix}' for signal in key_signals]
        group = (pooled_q_auc[pooled_q_auc.planner.eq(planner)]
                 .set_index('score_name').reindex(names))
        y = np.arange(len(names))
        auc = group.failure_auc.to_numpy(float)
        ax.errorbar(auc, y,
            xerr=np.vstack((auc - group.auc_ci_low.to_numpy(float),
                            group.auc_ci_high.to_numpy(float) - auc)),
            fmt='o', capsize=3, color='#4C78A8')
        ax.axvline(.5, color='black', linestyle='--')
        ax.set_yticks(y, key_signals)
        ax.set(xlim=(0, 1), xlabel='Failure ROC-AUC',
               title=f'{planner}: {suffix.replace("_", " ")}')
        ax.grid(axis='x', alpha=.2)
fig.suptitle('Failure prediction from Q/candidate diagnostics')
fig.savefig(OUTPUT / 'q_signal_pooled_auc.png', dpi=180)
plt.show()


In [ ]:
first_scores = [f'{signal}_first_chunk' for signal in key_signals]
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
for ax, method in zip(axes, (Method.QPLANNING_Q10, Method.QPLANNING_Q50)):
    planner = ARM_LABELS[method]
    group = q_features[q_features.method.eq(method)]
    failure = (~group.success.astype(bool)).astype(int).to_numpy()
    for score in first_scores:
        fpr, tpr, _ = roc_curve(failure, group[score].to_numpy(float))
        auc = pooled_q_auc[
            pooled_q_auc.planner.eq(planner)
            & pooled_q_auc.score_name.eq(score)].failure_auc.iloc[0]
        ax.plot(fpr, tpr, label=f'{score.removesuffix("_first_chunk")}: {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', label='chance')
    ax.set(xlabel='False-positive rate', ylabel='True-positive rate',
           title=f'{planner}: first-boundary failure ROC')
    ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.savefig(OUTPUT / 'q_first_boundary_roc.png', dpi=180)
plt.show()


## First-boundary Q/candidate failure AUC by suite

These are the full #68 held-out suites. Wide intervals are expected with 20–30 episodes
per suite; use them to identify consistent direction, not to rank tiny differences.


In [ ]:
suite_q = q_auc[
    ~q_auc.suite.eq('pooled')
    & q_auc.score_name.isin(first_scores)].copy()
suite_order = sorted(suite_q.suite.unique())
fig, axes = plt.subplots(1, 2, figsize=(16, 7), constrained_layout=True)
offsets = np.linspace(-.24, .24, len(first_scores))
for ax, planner in zip(axes, ('Q10 planner', 'Q50 planner')):
    for offset, score, color in zip(
            offsets, first_scores,
            ('#4C78A8', '#F58518', '#54A24B', '#B279A2')):
        group = (suite_q[
            suite_q.planner.eq(planner) & suite_q.score_name.eq(score)]
            .set_index('suite').reindex(suite_order))
        valid = group.failure_auc.notna().to_numpy()
        auc = group.failure_auc.to_numpy(float)[valid]
        ax.errorbar(
            auc, np.arange(len(suite_order))[valid] + offset,
            xerr=np.vstack((auc - group.auc_ci_low.to_numpy(float)[valid],
                            group.auc_ci_high.to_numpy(float)[valid] - auc)),
            fmt='o', capsize=2, color=color,
            label=score.removesuffix('_first_chunk'))
    ax.set_yticks(np.arange(len(suite_order)),
                  [s.removeprefix('libero_') for s in suite_order])
    ax.axvline(.5, color='black', linestyle='--')
    ax.set(xlim=(0, 1), xlabel='Failure ROC-AUC (95% bootstrap CI)',
           title=planner)
    ax.legend(fontsize=7); ax.grid(axis='x', alpha=.2)
fig.suptitle('First-boundary Q/candidate failure AUC by held-out suite')
fig.savefig(OUTPUT / 'q_first_boundary_auc_by_suite.png', dpi=180)
plt.show()


## Prefix and exact-chunk tests

The prefix plot retains every planner episode in each denominator and averages however
many of the first k chunks that episode produced. The exact-chunk plot includes only
episodes that reached that chunk, so later points are survivor-biased and descriptive.


In [ ]:
prefix_auc = q_prefix_auc_table(q_prefix, n_boot=N_BOOT)
chunk_auc = q_chunk_auc_table(boundaries, n_boot=N_BOOT, min_episodes=20)
prefix_auc.to_csv(OUTPUT / 'q_first_k_chunk_auc.csv', index=False)
chunk_auc.to_csv(OUTPUT / 'q_exact_chunk_auc.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
for row_idx, method in enumerate((Method.QPLANNING_Q10, Method.QPLANNING_Q50)):
    planner = ARM_LABELS[method]
    for signal in key_signals:
        group = prefix_auc[
            prefix_auc.method.eq(method) & prefix_auc.score_name.eq(signal)]
        axes[row_idx, 0].plot(group.first_k_chunks, group.failure_auc,
                              marker='o', label=signal)
        exact = chunk_auc[
            chunk_auc.method.eq(method) & chunk_auc.score_name.eq(signal)]
        axes[row_idx, 1].plot(exact.chunk_idx + 1, exact.failure_auc,
                              marker='o', label=signal)
    axes[row_idx, 0].axhline(.5, color='black', linestyle='--')
    axes[row_idx, 0].set(xlabel='First k chunks averaged', ylabel='Failure ROC-AUC',
                          ylim=(.25, 1), title=f'{planner}: prefix signal')
    axes[row_idx, 1].axhline(.5, color='black', linestyle='--')
    axes[row_idx, 1].set(xlabel='Exact chunk index (1-based)',
                          ylabel='Failure ROC-AUC', ylim=(.25, 1),
                          title=f'{planner}: exact chunk (survivor-biased)')
    for ax in axes[row_idx]:
        ax.legend(fontsize=7); ax.grid(alpha=.2)
fig.savefig(OUTPUT / 'q_prefix_and_chunk_auc.png', dpi=180)
plt.show()


## Candidate-diversity profiles by eventual outcome

This tests whether failures are associated with more disagreement among the 64 candidates,
and whether that disagreement is concentrated in the first 10 executable positions or the
unexecuted tail.


In [ ]:
profile = (boundaries.groupby(['planner', 'success', 'chunk_idx'])[
    ['first10_pairwise_rms', 'full50_pairwise_rms']]
    .agg(['mean', 'sem']).reset_index())
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for ax, planner in zip(axes, ('Q10 planner', 'Q50 planner')):
    group = profile[profile.planner.eq(planner)]
    for success, linestyle, label in ((True, '-', 'success'), (False, '--', 'failure')):
        selected = group[group.success.eq(success)]
        for metric, color, horizon in (
                ('first10_pairwise_rms', '#4C78A8', 'first 10'),
                ('full50_pairwise_rms', '#F58518', 'full 50')):
            ax.plot(selected.chunk_idx + 1, selected[(metric, 'mean')],
                    linestyle=linestyle, color=color, marker='o',
                    label=f'{label}, {horizon}')
    ax.set(xlabel='Chunk index (1-based)', ylabel='Mean pairwise candidate RMS',
           title=planner)
    ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.savefig(OUTPUT / 'candidate_diversity_by_outcome.png', dpi=180)
plt.show()


## Load true U10/U20/U50 and contraction on the worker-41 overlap

This is a separate deterministic stock replay on the same physical identities, not telemetry
from workers 68. We first report replay/current-stock outcome agreement. The overlap is 140:
x0.1, y0.1, x0.2, y0.2, and y0.3; x0.3 was absent from worker 41.


In [ ]:
u_features = u_records = u_positions = u_iterations = None
if INCLUDE_WORKER41_U:
    diagnostic_config = RolloutConfig(
        pnp_steps=(3, 4), pnp_k=5, n_action_steps=10)
    diagnostic_hash = store.config_hash(
        store._logical_key(Method.UNCERTAINTY, diagnostic_config))
    diagnostic_rows = pd.DataFrame(store.fetch_all(
        'rollouts', '*', configure=lambda query: query.eq(
            'experiment', SOURCE_HORIZON_MULTI_QUERY_EXPERIMENT).eq(
            'method', Method.UNCERTAINTY).eq('config_hash', diagnostic_hash),
        order_by=('rollout_id',)))
    diagnostic_rows = validate_diagnostic_cohort(
        diagnostic_rows, expected_identities=1300, require_complete=False)
    stock_keys = set(map(tuple, arms[Method.VANILLA][DIVERSITY_PAIR_KEYS]
                         .itertuples(index=False, name=None)))
    diagnostic_rows = diagnostic_rows[
        diagnostic_rows[DIVERSITY_PAIR_KEYS].apply(tuple, axis=1).isin(stock_keys)]
    if len(diagnostic_rows) != EXPECTED_U_OVERLAP:
        raise ValueError(
            f'expected {EXPECTED_U_OVERLAP} exact worker-41 overlaps, '
            f'found {len(diagnostic_rows)}')
    cache_files = {
        name: CACHE / f'worker41_overlap_{name}.pkl'
        for name in ('features', 'records', 'positions', 'iterations')}
    if all(path.exists() for path in cache_files.values()):
        u_features = pd.read_pickle(cache_files['features'])
        u_records = pd.read_pickle(cache_files['records'])
        u_positions = pd.read_pickle(cache_files['positions'])
        u_iterations = pd.read_pickle(cache_files['iterations'])
        print('Loaded worker-41 overlap artifacts from local cache.')
    else:
        u_features, u_records, u_positions, u_iterations = load_horizon_artifacts(
            store, diagnostic_rows, progress=tqdm)
        for name, frame in zip(cache_files,
                (u_features, u_records, u_positions, u_iterations)):
            frame.to_pickle(cache_files[name])

    stock_outcomes = arms[Method.VANILLA][
        DIVERSITY_PAIR_KEYS + ['success']].rename(columns={
            'success': 'current_stock_success'})
    u_features = u_features.rename(columns={'success': 'worker41_replay_success'})
    u_features = u_features.merge(
        stock_outcomes, on=DIVERSITY_PAIR_KEYS, validate='one_to_one')
    u_features['success'] = u_features.current_stock_success.astype(bool)
    u_features['worker41_replay_success'] = (
        u_features.worker41_replay_success.astype(bool))
    outcome_match = u_features.worker41_replay_success.eq(
        u_features.current_stock_success).mean()
    print({'U_overlap_identities': len(u_features),
           'current_stock_vs_worker41_outcome_agreement_pct': 100 * outcome_match,
           'missing_suite': 'libero_object_temp_x0.3'})
    display(u_features.groupby('suite').agg(
        episodes=('rollout_id', 'size'),
        current_stock_sr=('current_stock_success', 'mean'),
        worker41_replay_sr=('worker41_replay_success', 'mean')))


## True P&P U failure AUC, prefix/chunk study, contraction, and action position

All AUC labels here use the current stock outcome on the 140-identity overlap. Per-position
U is available because worker 41 saved a 50-position profile. Per-action-dimension U is **not**
available: the recorder reduced across the 7 action dimensions before saving `u_time`.


In [ ]:
if u_features is not None:
    u_episode_scores = [f'u{h}_episode' for h in HORIZONS]
    u_first_scores = [f'u{h}_first_chunk' for h in HORIZONS]
    u_auc = failure_auc_table(
        u_features, u_episode_scores + u_first_scores, n_boot=N_BOOT)
    display(u_auc[u_auc.suite.eq('pooled')][[
        'score_name', 'episodes', 'failures', 'failure_auc',
        'auc_ci_low', 'auc_ci_high']])
    u_auc.to_csv(OUTPUT / 'worker41_overlap_u_failure_auc.csv', index=False)

    per_suite = u_auc[
        ~u_auc.suite.eq('pooled') & u_auc.score_name.isin(u_episode_scores)]
    suite_order = sorted(per_suite.suite.unique())
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)
    for offset, score, color in zip((-.18, 0, .18), u_episode_scores,
                                     ('#4C78A8', '#F58518', '#54A24B')):
        group = per_suite[per_suite.score_name.eq(score)].set_index(
            'suite').reindex(suite_order)
        valid = group.failure_auc.notna().to_numpy()
        auc = group.failure_auc.to_numpy(float)[valid]
        axes[0].errorbar(auc, np.arange(len(suite_order))[valid] + offset,
            xerr=np.vstack((auc - group.auc_ci_low.to_numpy(float)[valid],
                            group.auc_ci_high.to_numpy(float)[valid] - auc)),
            fmt='o', capsize=2, color=color, label=score)
    axes[0].set_yticks(np.arange(len(suite_order)),
                       [s.removeprefix('libero_') for s in suite_order])
    axes[0].axvline(.5, color='black', linestyle='--')
    axes[0].set(xlim=(0, 1), xlabel='Failure ROC-AUC (95% bootstrap CI)',
                title='True P&P uncertainty AUC by suite')
    axes[0].legend(fontsize=8); axes[0].grid(axis='x', alpha=.2)
    failure = (~u_features.success).astype(int).to_numpy()
    pooled = u_auc[u_auc.suite.eq('pooled')]
    for score in u_episode_scores:
        fpr, tpr, _ = roc_curve(failure, u_features[score].to_numpy(float))
        auc = pooled[pooled.score_name.eq(score)].failure_auc.iloc[0]
        axes[1].plot(fpr, tpr, label=f'{score}: {auc:.3f}')
    axes[1].plot([0, 1], [0, 1], 'k--', label='chance')
    axes[1].set(xlabel='False-positive rate', ylabel='True-positive rate',
                title='Pooled true-U failure ROC')
    axes[1].legend(); axes[1].grid(alpha=.2)
    fig.savefig(OUTPUT / 'worker41_overlap_u_auc_and_roc.png', dpi=180)
    plt.show()


In [ ]:
if u_features is not None:
    current_outcomes = u_features[['rollout_id', 'success']]
    u_records = (u_records.drop(columns=['success'], errors='ignore')
                 .merge(current_outcomes, on='rollout_id', validate='many_to_one'))
    u_positions = (u_positions.drop(columns=['success'], errors='ignore')
                   .merge(current_outcomes, on='rollout_id', validate='many_to_one'))
    u_iterations = (u_iterations.drop(columns=['success'], errors='ignore')
                    .merge(current_outcomes, on='rollout_id', validate='many_to_one'))
    u_prefix = prefix_feature_table(u_records, u_features, max_chunks=8)
    u_prefix_auc = prefix_failure_auc_table(u_prefix, n_boot=N_BOOT)
    u_prefix_auc.to_csv(OUTPUT / 'worker41_overlap_first_k_auc.csv', index=False)

    exact_metric_columns = ([f'u{h}' for h in HORIZONS]
                            + [f'contraction{h}' for h in HORIZONS])
    u_exact_records = (u_records.groupby(
        ['rollout_id', 'suite', 'success', 'chunk_idx'], sort=True)
        [exact_metric_columns].mean().reset_index())
    exact_rows = []
    for chunk_idx, group in u_exact_records.groupby('chunk_idx', sort=True):
        if len(group) < 20:
            continue
        scores = ([f'u{h}' for h in HORIZONS]
                  + [f'negative_contraction{h}' for h in HORIZONS])
        group = group.copy()
        for horizon in HORIZONS:
            group[f'negative_contraction{horizon}'] = -group[f'contraction{horizon}']
        table = failure_auc_table(
            group, scores, by_suite=False, n_boot=N_BOOT)
        table['chunk_idx'] = int(chunk_idx)
        table['episodes_reaching_chunk'] = len(group)
        exact_rows.append(table)
    u_exact_auc = pd.concat(exact_rows, ignore_index=True)
    u_exact_auc.to_csv(OUTPUT / 'worker41_overlap_exact_chunk_auc.csv', index=False)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
    for horizon, color in zip(HORIZONS, ('#4C78A8', '#F58518', '#54A24B')):
        group = u_prefix_auc[
            u_prefix_auc.score_type.eq('uncertainty')
            & u_prefix_auc.action_horizon.eq(horizon)]
        axes[0, 0].plot(group.first_k_chunks, group.failure_auc,
                        marker='o', color=color, label=f'U{horizon}')
        contraction = u_prefix_auc[
            u_prefix_auc.score_type.eq('negative_contraction')
            & u_prefix_auc.action_horizon.eq(horizon)]
        axes[0, 1].plot(contraction.first_k_chunks, contraction.failure_auc,
                        marker='o', color=color, label=f'contraction {horizon}')
        exact = u_exact_auc[u_exact_auc.score_name.eq(f'u{horizon}')]
        axes[1, 0].plot(exact.chunk_idx + 1, exact.failure_auc,
                        marker='o', color=color, label=f'U{horizon}')
        exact_c = u_exact_auc[
            u_exact_auc.score_name.eq(f'negative_contraction{horizon}')]
        axes[1, 1].plot(exact_c.chunk_idx + 1, exact_c.failure_auc,
                        marker='o', color=color, label=f'contraction {horizon}')
    titles = (
        'Prefix uncertainty predicts failure',
        'Prefix weak/non-contraction predicts failure',
        'Exact-chunk uncertainty (survivor-biased)',
        'Exact-chunk weak/non-contraction (survivor-biased)')
    for ax, title in zip(axes.flat, titles):
        ax.axhline(.5, color='black', linestyle='--')
        ax.set(xlabel=('First k chunks averaged' if 'Prefix' in title
                       else 'Exact chunk index (1-based)'),
               ylabel='Failure ROC-AUC', ylim=(.25, 1), title=title)
        ax.legend(); ax.grid(alpha=.2)
    fig.savefig(OUTPUT / 'worker41_overlap_prefix_and_exact_chunk_auc.png', dpi=180)
    plt.show()

    position_summary = (u_positions.groupby(['success', 'action_position'])
        .uncertainty.agg(['mean', 'sem']).reset_index())
    fig, ax = plt.subplots(figsize=(12, 5))
    for success, group in position_summary.groupby('success', sort=False):
        color = '#4C78A8' if bool(success) else '#E45756'
        label = 'successful stock episodes' if bool(success) else 'failed stock episodes'
        ax.plot(group.action_position, group['mean'], color=color, label=label)
        ax.fill_between(group.action_position, group['mean'] - group['sem'],
                        group['mean'] + group['sem'], color=color, alpha=.16)
    ax.axvline(9.5, color='black', linestyle='--', label='10 executed actions')
    ax.axvline(19.5, color='#9467BD', linestyle=':', label='U20 boundary')
    ax.set(xlabel='Action position in generated 50-action chunk',
           ylabel='Mean uncertainty',
           title='True P&P uncertainty by action position and outcome')
    ax.legend(); ax.grid(alpha=.2)
    fig.tight_layout(); fig.savefig(OUTPUT / 'worker41_overlap_u_by_position.png', dpi=180)
    plt.show()
    print('Per-action-dimension U: unavailable (not persisted by workers 41 or 68).')


## Exploratory gate screening

For each planner, this asks: if we chose stock or that planner using a signal, is there a
threshold/window with more F→S than S→F outcomes? Every proxy delta uses all 140 overlapping
identities in its denominator.

- `*_first_chunk`: an episode-start rule could compute this before acting, then commit to one arm.
- `*_episode`: post-hoc only.
- This does not simulate switching at every chunk. A real per-boundary gate requires a new online run.


In [ ]:
gate_sweeps, selector_rows = [], []
if u_features is not None:
    gate_scores = ([f'u{h}_first_chunk' for h in HORIZONS]
                   + [f'u{h}_episode' for h in HORIZONS])
    for method in (Method.QPLANNING_Q10, Method.QPLANNING_Q50):
        for score in gate_scores:
            pair = attach_signal_to_pair(pairs[method], u_features, score)
            sweep = quantile_gate_sweep(
                pair, score_column=score, grid_size=GATE_GRID_SIZE,
                min_selected=MIN_GATE_EPISODES)
            sweep.insert(0, 'planner', ARM_LABELS[method])
            gate_sweeps.append(sweep)
            selector = discordant_selector_auc(pair, score_column=score, n_boot=N_BOOT)
            selector['planner'] = ARM_LABELS[method]
            selector['interpretation'] = (
                'episode-start proxy' if score.endswith('first_chunk') else 'post-hoc only')
            selector_rows.append(selector)
    gate_sweeps = pd.concat(gate_sweeps, ignore_index=True)
    selector_auc = pd.DataFrame(selector_rows)
    eligible = gate_sweeps[gate_sweeps.eligible].copy()
    best_gates = (eligible.sort_values(
        ['planner', 'score_name', 'gate_kind', 'proxy_delta_pp', 'episodes_selected'],
        ascending=[True, True, True, False, False])
        .groupby(['planner', 'score_name', 'gate_kind'], sort=False).head(1))
    best_gates['interpretation'] = np.where(
        best_gates.score_name.str.endswith('first_chunk'),
        'episode-start proxy', 'post-hoc only')
    display(best_gates[[
        'planner', 'score_name', 'interpretation', 'gate_kind',
        'episodes_in_sr_denominator', 'lower', 'upper', 'episodes_selected',
        'stock_sr_pct', 'proxy_policy_sr_pct', 'proxy_delta_pp',
        'selected_F_to_S', 'selected_S_to_F']])
    print('Can the signal rank Q wins above Q losses on discordant episodes?')
    display(selector_auc[[
        'planner', 'score_name', 'interpretation', 'discordant_episodes',
        'Q_wins', 'Q_losses', 'Q_win_auc', 'auc_ci_low', 'auc_ci_high']])
    gate_sweeps.to_csv(OUTPUT / 'u_gate_sweeps.csv', index=False)
    best_gates.to_csv(OUTPUT / 'best_u_gates.csv', index=False)
    selector_auc.to_csv(OUTPUT / 'u_Q_win_auc.csv', index=False)


In [ ]:
if u_features is not None:
    for suffix, title in (('first_chunk', 'Episode-start U-window screen'),
                          ('episode', 'Post-hoc whole-episode U-window screen')):
        fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
        for row_idx, planner in enumerate(('Q10 planner', 'Q50 planner')):
            for col_idx, horizon in enumerate(HORIZONS):
                score = f'u{horizon}_{suffix}'
                sweep = gate_sweeps[
                    gate_sweeps.planner.eq(planner)
                    & gate_sweeps.score_name.eq(score)
                    & gate_sweeps.gate_kind.eq('bounded_window')
                    & gate_sweeps.eligible]
                pivot = sweep.pivot_table(
                    index='lower', columns='upper', values='proxy_delta_pp',
                    aggfunc='max').sort_index()
                limit = max(1, np.nanmax(np.abs(pivot.to_numpy())))
                image = axes[row_idx, col_idx].imshow(
                    pivot.to_numpy(), aspect='auto', origin='lower',
                    cmap='RdYlGn', vmin=-limit, vmax=limit)
                yi = np.linspace(0, len(pivot.index) - 1,
                                 min(5, len(pivot.index))).astype(int)
                xi = np.linspace(0, len(pivot.columns) - 1,
                                 min(5, len(pivot.columns))).astype(int)
                axes[row_idx, col_idx].set_yticks(
                    yi, [f'{pivot.index[i]:.3f}' for i in yi])
                axes[row_idx, col_idx].set_xticks(
                    xi, [f'{pivot.columns[i]:.3f}' for i in xi], rotation=30)
                axes[row_idx, col_idx].set(
                    xlabel='Upper U bound', ylabel='Lower U bound',
                    title=f'{planner}, U{horizon}')
                fig.colorbar(image, ax=axes[row_idx, col_idx],
                             label='SR proxy change (pp), all 140 episodes')
        fig.suptitle(title)
        fig.savefig(OUTPUT / f'u_window_heatmaps_{suffix}.png', dpi=180)
        plt.show()

    first_thresholds = gate_sweeps[
        gate_sweeps.gate_kind.eq('high_threshold')
        & gate_sweeps.score_name.str.endswith('first_chunk')
        & gate_sweeps.eligible]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
    for ax, planner in zip(axes, ('Q10 planner', 'Q50 planner')):
        for horizon, color in zip(HORIZONS, ('#4C78A8', '#F58518', '#54A24B')):
            group = first_thresholds[
                first_thresholds.planner.eq(planner)
                & first_thresholds.score_name.eq(f'u{horizon}_first_chunk')]
            ax.plot(group.lower, group.proxy_delta_pp, marker='.', color=color,
                    label=f'U{horizon}')
        ax.axhline(0, color='black', linewidth=1)
        ax.set(xlabel='Choose planner when first-chunk U ≥ threshold',
               ylabel='SR proxy change vs stock (pp)', title=planner)
        ax.legend(); ax.grid(alpha=.2)
    fig.savefig(OUTPUT / 'u_first_chunk_threshold_sweeps.png', dpi=180)
    plt.show()


## Compact decision summary

Use the paired SR table to decide whether either planner merits further work. For gating,
prioritize first-chunk Q/U AUC and first-chunk Q-win AUC. Strong whole-episode AUC alone is
useful for training targets or failure detection, but cannot support an online deployment
claim without a new rollout policy. If per-action-dimension U is needed, rerun a small no-op
diagnostic that saves unreduced action-dimension disagreement; it is absent from these logs.


In [ ]:
print('Matched three-arm identities:', n_matched)
print('\nPrimary paired effects:')
display(effects[[
    'planner', 'episodes', 'baseline_sr_pct', 'condition_sr_pct',
    'condition_minus_baseline_pp', 'delta_ci_low_pp', 'delta_ci_high_pp',
    'failure_to_success', 'success_to_failure']])
print('\nStrongest pooled first-boundary Q/candidate failure signals:')
display(pooled_q_auc[
    pooled_q_auc.score_name.str.endswith('first_chunk')]
    .sort_values('failure_auc', ascending=False)
    [['planner', 'score_name', 'failure_auc', 'auc_ci_low', 'auc_ci_high']]
    .groupby('planner').head(5))
if u_features is not None:
    print('\nTrue-U overlap:', len(u_features),
          '(all suites except object_temp_x0.3)')
    display(u_auc[u_auc.suite.eq('pooled')][[
        'score_name', 'failure_auc', 'auc_ci_low', 'auc_ci_high']])
    print('\nBest episode-start gate screens (exploratory; all 140 in denominator):')
    display(best_gates[
        best_gates.interpretation.eq('episode-start proxy')]
        .sort_values('proxy_delta_pp', ascending=False)[[
            'planner', 'score_name', 'gate_kind', 'episodes_selected',
            'proxy_policy_sr_pct', 'proxy_delta_pp']].head(12))
print('\nOutputs saved to:', OUTPUT.resolve())
